# From Red Flags to Detection Rules
## An LLM-driven Pipeline for Real-Time GOOSE Intrusion Detection and Prevention

> **Autores:** Lucas A. Martins¹*, Silvio E. Quincozes¹²  
> ¹ Universidade Federal de Uberlandia (UFU) – Uberlandia, Brasil  
> ² Universidade Federal do Pampa (UNIPAMPA) – Alegrete, Brasil  
> `{lucas.martins, sequincozes}@ufu.br`
>
> Artigo submetido ao **XLIV SBRC 2026**

---

### Resumo

Sistemas de Deteccao de Intrusao (IDS) baseados em especificacao sao amplamente utilizados
em subestacoes IEC 61850, mas dependem de regras criadas manualmente por especialistas.
Este notebook apresenta um **pipeline orientado por LLM** que automatiza a geracao de
regras de deteccao para deteccao e prevencao de intrusoes GOOSE em tempo real.

A abordagem utiliza amostras de comunicacao rotuladas do dataset **ERENO-2.0** para
identificar *red flags*, transformadas em regras Python executaveis e testadas em ambiente
de switch programavel simulado. Os resultados mostram deteccao com **baixo overhead
operacional** (P99 < 5 us por pacote), indicando que a automacao via LLM e viavel para
IDS baseados em especificacao.

## Indice

1. [Introducao](#1-introducao)
2. [IDS Baseado em Especificacao para GOOSE](#2-ids-baseado-em-especificacao-para-goose)
3. [Arquitetura Proposta](#3-arquitetura-proposta)
4. [Instalacao e Configuracao do Ambiente](#4-instalacao-e-configuracao-do-ambiente)
5. [Ingestao de Dados - ERENO-2.0](#5-ingestao-de-dados--ereno-20)
6. [Extracao de Red Flags via LLM](#6-extracao-de-red-flags-via-llm)
7. [Geracao de Regras de Deteccao](#7-geracao-de-regras-de-deteccao)
8. [Simulacao do Switch Programavel](#8-simulacao-do-switch-programavel)
9. [Avaliacao - Capacidade de Deteccao](#9-avaliacao--capacidade-de-deteccao)
10. [Avaliacao - Viabilidade em Tempo Real](#10-avaliacao--viabilidade-em-tempo-real)
11. [Consideracoes Finais](#11-consideracoes-finais)
12. [Referencias](#12-referencias)

---
## 1. Introducao

Subestacoes digitais baseadas no padrao **IEC-61850** enfrentam desafios crescentes de
ciberseguranca, incluindo ataques de:

- **Denial-of-Service (DoS)**: inundacao de mensagens para sobrecarregar canal ou dispositivo
- **Message Injection**: insercao de mensagens falsas com campos manipulados
- **Masquerade**: impersonacao de dispositivos legitimos com campos fabricados
- **Replay**: retransmissao de mensagens validas capturadas anteriormente

IDS baseados em especificacao sao atraentes por seu **baixo overhead computacional** e
**interpretabilidade**. No entanto, dependem de regras escritas manualmente, processo
custoso e pouco adaptavel a novos padroes de ataque [Quincozes et al. 2021].

**Motivacao principal:** Automatizar a geracao dessas regras usando LLMs a partir de
amostras rotuladas do dataset **ERENO-IEC-61850 v2.0**.

> **Lacunas endereçadas neste trabalho:**  
> - Sec. 5 (Prova de Conceito): metricas de deteccao por classe e por regra com resultados experimentais  
> - Sec. 5.2 (Viabilidade em Tempo Real): benchmarks de latencia medidos experimentalmente no ERENO-2.0

---
## 2. IDS Baseado em Especificacao para GOOSE

O protocolo **GOOSE (Generic Object Oriented Substation Event)**, definido pelo padrao
IEC-61850-8-1, suporta operacoes de protecao e controle em tempo critico via modelo
publisher/subscriber sobre Ethernet.

### Campos relevantes de um frame GOOSE

| Categoria | Campos |
|-----------|--------|
| Estruturais | `ethDst`, `TPID`, `ethType`, `gooseAppid`, `gooseTimeAllowedtoLive` |
| Consistencia | `gocbRef`, `datSet`, `goID`, `t`, `StNum`, `SqNum`, `cbStatus` |
| Derivados/Temporais | `stDiff`, `sqDiff`, `timestampDiff`, `tDiff`, `timeFromLastChange`, `delay` |
| Payload | `gooseLen`, `APDUSize`, `frameLen` |

> O GOOSE nao foi projetado com mecanismos nativos de seguranca robustos, tornando-o
> vulneravel a ataques de injecao, replay e negacao de servico [Malik et al. 2022].

---
## 3. Arquitetura Proposta

O pipeline e composto por **quatro estagios principais**:

```
+------------------+    +------------------+    +------------------+    +------------------+
|  Dataset GOOSE   |    |  Red Flag Extr.  |    |  Rule Generation |    | Switch Simulation|
|  Rotulado        |--->|  (LLM-based)     |--->|  (Python rules)  |--->|  (Real-time)     |
|  (ERENO-2.0)     |    |                  |    |                  |    |                  |
+------------------+    +------------------+    +------------------+    +------------------+
```

| Estagio | Responsabilidade |
|---------|------------------|
| **1. Source Ingestion** | Carrega dataset ERENO-2.0, organiza features, prepara prompts |
| **2. Red Flag Extraction** | LLM identifica padroes suspeitos e inconsistencias comportamentais |
| **3. Rule Generation** | Traduz red flags em regras Python executaveis por classe de ataque |
| **4. Simulated Deployment** | Executa regras sobre trafego GOOSE em ambiente de switch simulado |

---
## 4. Instalacao e Configuracao do Ambiente

### Pre-requisitos

- Python 3.9+
- Chave de API Groq (obtenha gratuitamente em https://console.groq.com)
- Arquivo `.env` com `GROQ_API_KEY=<sua_chave>` na mesma pasta do notebook
- Dataset `ERENO-2_0-1K_test.csv` na mesma pasta do notebook

> **Sem chave API:** o notebook funciona completamente usando as regras pre-geradas embutidas na Secao 7.

In [1]:
# Instalacao das dependencias
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'groq>=1.1.2', 'python-dotenv>=1.0.0',
     'pandas>=2.0.0', 'numpy>=1.24.0',
     'scikit-learn>=1.3.0', '--quiet'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('Dependencias instaladas com sucesso.')
else:
    print('Aviso na instalacao:', result.stderr[:300])

Dependencias instaladas com sucesso.


In [2]:
# Imports e configuracao da API
import os, time, json, re, importlib, importlib.util, sys
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
from groq import Groq, RateLimitError
from dotenv import load_dotenv

load_dotenv()   # le .env na pasta atual

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
GROQ_MODEL   = 'compound-beta'           # ou 'llama-3.3-70b-versatile'
DATASET_PATH = Path('ERENO-2.0-1K_test.csv')
RULES_FILE   = Path('rules_generated.py')

def get_groq_client() -> Groq:
    if not GROQ_API_KEY:
        raise RuntimeError(
            'Defina GROQ_API_KEY no arquivo .env\n'
            'Obtenha sua chave em https://console.groq.com'
        )
    return Groq(api_key=GROQ_API_KEY)

RUN_LLM = bool(GROQ_API_KEY)

print('Ambiente configurado.')
print(f'  Modelo LLM     : {GROQ_MODEL}')
print(f'  Dataset        : {DATASET_PATH}')
print(f'  Rules output   : {RULES_FILE}')
print(f'  LLM disponivel : {RUN_LLM}')

Ambiente configurado.
  Modelo LLM     : compound-beta
  Dataset        : ERENO-2.0-1K_test.csv
  Rules output   : rules_generated.py
  LLM disponivel : False


---
## 5. Ingestao de Dados - ERENO-2.0

O dataset **ERENO-IEC-61850 v2.0** fornece amostras rotuladas de trafego GOOSE em
condicoes normais e sob ataque. Cada amostra contem 52 features em tres grupos:

- **Nivel de protocolo:** campos nativos do PDU GOOSE (`StNum`, `SqNum`, `cbStatus`, ...)
- **Temporais:** diferencas entre mensagens consecutivas (`stDiff`, `sqDiff`, `timestampDiff`, `tDiff`)
- **Derivadas:** metricas de comportamento (`timeFromLastChange`, `delay`, `frameLengthDiff`)

### Classes de trafego no dataset (207 amostras balanceadas, 23 por classe)

| Classe | Tipo | Descricao |
|--------|------|-----------|
| `normal` | Benigno | Trafego legitimo de subestacao |
| `grayhole` | DoS | Descarte seletivo de mensagens |
| `high_StNum` | Injection | StNum artificialmente elevado |
| `injection` | Injection | Insercao de mensagens com campos manipulados |
| `inverse_replay` | Replay | Reproducao invertida de sequencia |
| `masquerade_fake_fault` | Masquerade | Impersonacao com falha falsa |
| `masquerade_fake_normal` | Masquerade | Impersonacao simulando operacao normal |
| `poisoned_high_rate` | DoS | Flood de mensagens envenenadas |
| `random_replay` | Replay | Reproducao aleatoria de mensagens |

In [3]:
# 5.1 Carregamento do dataset
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset nao encontrado: {DATASET_PATH.resolve()}\n'
        'Coloque o arquivo ERENO-2_0-1K_test.csv na mesma pasta deste notebook.'
    )

df = pd.read_csv(DATASET_PATH)

print(f'Dataset carregado: {df.shape[0]} amostras x {df.shape[1]} features')
print()
print('Distribuicao de classes:')
vc = df['class'].value_counts()
for cls, cnt in vc.items():
    bar = '#' * cnt
    print(f'  {cls:30s}  {cnt:4d}  {bar}')

print()
cols_show = ['class','StNum','SqNum','stDiff','sqDiff','timestampDiff','tDiff','timeFromLastChange','delay']
display(df[cols_show].head(6))

Dataset carregado: 207 amostras x 52 features

Distribuicao de classes:
  poisoned_high_rate                23  #######################
  grayhole                          23  #######################
  inverse_replay                    23  #######################
  masquerade_fake_fault             23  #######################
  masquerade_fake_normal            23  #######################
  normal                            23  #######################
  high_StNum                        23  #######################
  random_replay                     23  #######################
  injection                         23  #######################



,class,StNum,SqNum,stDiff,sqDiff,timestampDiff,tDiff,timeFromLastChange,delay
0,poisoned_high_rate,122.0,1.0,-5850.0,-80.0,0.116166,-805.919857,707.598591,0.000180
1,grayhole,287.0,2.0,0.0,0.0,0.000000,0.000000,0.004000,0.000203
2,inverse_replay,275.0,16.0,0.0,0.0,0.003353,0.000000,1318.944666,0.000195
3,masquerade_fake_fault,40.0,3.0,0.0,1.0,0.004000,0.000000,0.008000,0.000028
4,poisoned_high_rate,9773.0,1.0,1.0,0.0,6.438091,0.000000,58564.134007,0.000684
5,grayhole,499.0,39.0,-58974.0,7.0,0.258738,-34.445497,34.704236,0.000487


In [5]:
# 5.2 Analise estatistica por classe
NUMERIC_FEATURES = [
    'StNum','SqNum','stDiff','sqDiff',
    'timestampDiff','tDiff','timeFromLastChange','delay',
    'cbStatusDiff','gooseLengthDiff','apduSizeDiff','frameLengthDiff'
]
numeric_available = [c for c in NUMERIC_FEATURES if c in df.columns]

print('Mediana por classe (features numericas discriminativas):')
stats = df.groupby('class')[numeric_available].median().round(4)
display(stats)

IMPORTANT_COLUMNS = [
    'SqNum','StNum','cbStatus','gooseLen','APDUSize','frameLen',
    'gooseTimeAllowedtoLive','gooseAppid','TPID','ethType',
    'gocbRef','datSet','goID','test','confRev','ndsCom','numDatSetEntries',
    'stDiff','sqDiff','gooseLengthDiff','cbStatusDiff','apduSizeDiff',
    'frameLengthDiff','timestampDiff','tDiff','timeFromLastChange','delay',
    'class'
]
available_cols = [c for c in IMPORTANT_COLUMNS if c in df.columns]
df_reduced = df[available_cols].copy()
print(f'\nDataFrame reduzido para prompts: {df_reduced.shape[0]} linhas x {df_reduced.shape[1]} colunas')

Mediana por classe (features numericas discriminativas):


,StNum,SqNum,stDiff,sqDiff,timestampDiff,tDiff,timeFromLastChange,delay,cbStatusDiff,gooseLengthDiff,apduSizeDiff,frameLengthDiff
class,,,,,,,,,,,,
grayhole,325.0,10.0,0.0,0.0,0.0040,0.0000,5.2572,0.0001,0.0,0.0,0.0,0.0
high_StNum,45344.0,45.0,37383.0,22.0,0.0942,27.7331,0.0000,0.0002,0.0,0.0,0.0,0.0
injection,52.0,27.0,-176.0,0.0,0.0668,98.2702,0.0000,0.0002,1.0,0.0,0.0,0.0
inverse_replay,261.0,4.0,-83.0,-12.0,0.0176,-1160.2346,1417.5353,0.0002,1.0,0.0,0.0,0.0
masquerade_fake_fault,194.0,10.0,0.0,-1.0,0.0575,0.0000,5.2796,0.0001,0.0,0.0,0.0,0.0
masquerade_fake_normal,1096.0,3.0,82.0,-2.0,0.0063,3.2282,0.0080,0.0002,0.0,0.0,0.0,0.0
normal,339.0,14.0,2.0,1.0,0.0223,0.0000,9.3427,0.0001,0.0,0.0,0.0,0.0
poisoned_high_rate,5700.0,1.0,1.0,0.0,5.6294,0.0000,34217.9718,0.0007,0.0,0.0,0.0,0.0
random_replay,465.0,21.0,0.0,0.0,0.0550,-4.4000,16.6423,0.0001,0.0,0.0,0.0,0.0



DataFrame reduzido para prompts: 207 linhas x 28 colunas


---
## 6. Extracao de Red Flags via LLM

O LLM recebe amostras normais (referencia) e amostras de uma classe de ataque e e
instruido a **identificar padroes discriminativos** (red flags).

As red flags sao extraidas em linguagem natural semi-estruturada antes da geracao de
codigo, funcionando como camada intermediaria que melhora rastreabilidade e inspecao.

> Se voce nao possui chave Groq, as red flags pre-extraidas estao na Secao 7.

In [6]:
# 6.1 Funcoes de amostragem e prompt de red flags

def sample_normal_and_attacks(df, n_normal=20, n_per_attack=10):
    df_normal = df[df['class'] == 'normal'].sample(
        n=min(n_normal, (df['class'] == 'normal').sum()), random_state=42
    )
    attack_classes = sorted(c for c in df['class'].unique() if c != 'normal')
    attack_samples = {
        cls: df[df['class'] == cls].sample(
            n=min(n_per_attack, (df['class'] == cls).sum()), random_state=42
        )
        for cls in attack_classes
    }
    return df_normal, attack_samples


def make_red_flag_prompt(df_normal, df_attack, attack_class):
    header = (
        'Voce e um especialista em seguranca IEC 61850.\n'
        'Abaixo estao amostras de trafego GOOSE NORMAL (referencia legitima):\n\n'
        '=== AMOSTRAS NORMAIS ===\n'
        + df_normal.to_string(index=False)
    )
    attack_block = (
        f'\n\n=== AMOSTRAS DE ATAQUE: {attack_class} ===\n'
        + df_attack.to_string(index=False)
    )
    instructions = (
        f'\n\nCom base nas diferencas entre as amostras normais e as de ataque "{attack_class}",\n'
        'liste as RED FLAGS - condicoes numericas que distinguem este ataque do trafego normal.\n'
        'Formato: - [CAMPO]: condicao -> interpretacao\n'
        'Nao gere codigo Python nesta etapa.'
    )
    return header + attack_block + instructions


def call_llm_text(prompt, max_tokens=1024):
    client = get_groq_client()
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_completion_tokens=max_tokens,
    )
    return completion.choices[0].message.content.strip()


print('Funcoes de red flag definidas.')

Funcoes de red flag definidas.


In [7]:
# 6.2 Execucao da extracao de red flags (requer GROQ_API_KEY)
red_flags_extracted = {}

if RUN_LLM:
    df_normal_ref, attack_samples_rf = sample_normal_and_attacks(df_reduced)
    Path('logs').mkdir(exist_ok=True)

    for attack_class, df_attack in attack_samples_rf.items():
        print(f'\n--- Extraindo red flags: {attack_class} ---')
        prompt = make_red_flag_prompt(df_normal_ref, df_attack, attack_class)
        try:
            result = call_llm_text(prompt)
            red_flags_extracted[attack_class] = result
            print(result[:500], '...' if len(result) > 500 else '')
            time.sleep(1)
        except Exception as e:
            print(f'  [ERRO] {e}')

    Path('logs/red_flags.txt').write_text(
        '\n\n'.join(f'=== {k} ===\n{v}' for k, v in red_flags_extracted.items()),
        encoding='utf-8'
    )
    print('\nRed flags salvas em logs/red_flags.txt')
else:
    print('GROQ_API_KEY nao definida - use as regras pre-geradas na Secao 7.')

GROQ_API_KEY nao definida - use as regras pre-geradas na Secao 7.


---
## 7. Geracao de Regras de Deteccao

As red flags sao convertidas em **funcoes Python executaveis**. Cada funcao recebe
um pacote como dicionario e retorna `True` se o pacote for suspeito.

A abordagem gera multiplas regras por classe (individuais + `_combined`).

### Red flags pre-extraidas pelo pipeline (ERENO-2.0-1K_test)

```
GRAYHOLE
  - [StNum]: valor presente (>0) com SqNum=0 -> drop seletivo de retransmissoes
  - [stDiff/sqDiff]: saltos > 10 -> sequencia nao monotonica
  - [timestampDiff]: |diff| > 0.1 s -> inconsistencia temporal

HIGH_STNUM
  - [StNum]: valor > 50.000 -> manipulacao direta do campo de estado
  - [stDiff]: salto > 1.000 -> contador incrementado artificialmente
  - [timeFromLastChange]: < 1 s com StNum alto -> mudancas impossiveis

INJECTION
  - [stDiff]: negativo (< -10) -> StNum decrementado, impossivel em operacao normal
  - [sqDiff]: negativo (< -10) -> insercao de mensagem antiga
  - [tDiff]: > 0.1 -> gap temporal grande entre mensagens consecutivas

INVERSE_REPLAY
  - [stDiff]: negativo -> sequencia de estado invertida
  - [tDiff]: valores negativos grandes -> timestamps decrementando

MASQUERADE_FAKE_FAULT
  - [tDiff/timestampDiff]: |valor| > 0.1 -> timing inconsistente com fonte legitima
  - [stDiff/sqDiff]: |valor| > 10 -> valores fora do padrao de dispositivo real

MASQUERADE_FAKE_NORMAL
  - [StNum]: > 1.000 -> valor alto para dispositivo recem-iniciado
  - [sqDiff]: |valor| > 5.000 -> variacao de sequencia impossivel

POISONED_HIGH_RATE
  - [StNum]: > 1.000 com SqNum = 1 -> reinicio forcado com estado adulterado
  - [timestampDiff]: > 100 -> salto temporal indicando flood

RANDOM_REPLAY
  - [StNum - SqNum]: discrepancia > 10 -> contadores dessincronizados
  - [timestampDiff]: < 0.1 com tDiff > 0.1 -> replay por inconsistencia temporal
```

In [8]:
# 7.1 Funcoes de geracao de regras via LLM

def make_rule_gen_prompt(df_normal, df_attack, attack_class):
    header = (
        'Voce e um especialista em seguranca IEC 61850 e desenvolvedor Python.\n'
        'Amostras de trafego GOOSE NORMAL (referencia):\n\n=== AMOSTRAS NORMAIS ===\n'
        + df_normal.to_string(index=False)
    )
    attack_block = (
        f'\n\n=== AMOSTRAS DE ATAQUE: {attack_class} ===\n'
        + df_attack.to_string(index=False)
    )
    instructions = (
        f'\n\nGere funcoes DE REGRAS DE DETECCAO em Python para a classe "{attack_class}".\n'
        'Requisitos: retorne APENAS codigo Python valido, sem markdown.\n'
        f'Formato: def rule_{attack_class}_<nome>(packet: dict) -> bool:\n'
        '    # logica com .get(campo, 0)\n'
        f'Inclua obrigatoriamente rule_{attack_class}_combined.'
    )
    return header + attack_block + instructions


def call_llm_for_rules(prompt, max_retries=5):
    client = get_groq_client()
    system_msg = ('Voce e especialista em seguranca IEC 61850. '
                  'Retorne SOMENTE codigo Python valido, sem markdown.')
    attempt = 0
    while True:
        try:
            completion = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {'role': 'system', 'content': system_msg},
                    {'role': 'user',   'content': prompt},
                ],
                temperature=0, max_completion_tokens=2048,
            )
            code_text = completion.choices[0].message.content.strip()
            code_text = re.sub(r'^```(python)?\n?', '', code_text, flags=re.MULTILINE)
            code_text = re.sub(r'\n?```$', '', code_text, flags=re.MULTILINE)
            return code_text.strip()
        except RateLimitError as e:
            attempt += 1
            if attempt > max_retries: raise
            wait = min(60.0, 2 ** attempt)
            m = re.search(r'try again in ([\d.]+)s', str(e))
            if m: wait = float(m.group(1)) + 1
            print(f'  [RateLimit] aguardando {wait:.1f}s...')
            time.sleep(wait)


def append_to_rules_file(code_text, header='', filepath=None):
    if filepath is None: filepath = RULES_FILE
    mode = 'a' if filepath.exists() else 'w'
    with filepath.open(mode, encoding='utf-8') as f:
        if mode == 'a': f.write('\n\n')
        if header: f.write(f'## {header}\n\n')
        f.write(code_text.strip() + '\n')

print('Funcoes de geracao de regras definidas.')

Funcoes de geracao de regras definidas.


In [9]:
# 7.2 Execucao do pipeline de geracao de regras (requer GROQ_API_KEY)
if RUN_LLM:
    if RULES_FILE.exists(): RULES_FILE.unlink()
    df_normal_ref, attack_samples_rules = sample_normal_and_attacks(df_reduced)
    for attack_class, df_attack in attack_samples_rules.items():
        print(f'\n--- Gerando regras: {attack_class} ---')
        prompt = make_rule_gen_prompt(df_normal_ref, df_attack, attack_class)
        try:
            code_rules = call_llm_for_rules(prompt)
            append_to_rules_file(code_rules, header=attack_class)
            print(f'  Regras geradas ({len(code_rules)} chars)')
            time.sleep(2)
        except Exception as e:
            print(f'  [ERRO] {e}')
    print(f'\nRegras salvas em {RULES_FILE.resolve()}')
else:
    print('GROQ_API_KEY nao definida - celula 7.3 carregara as regras pre-geradas.')

GROQ_API_KEY nao definida - celula 7.3 carregara as regras pre-geradas.


In [10]:
# 7.3 Regras pre-geradas pelo pipeline LLM
# Geradas com compound-beta (Groq), 20 amostras normais + 10 por classe de ataque.
# Carregadas automaticamente se GROQ_API_KEY nao estiver disponivel.

RULES_CODE_EMBEDDED = "## grayhole\n\n\ndef rule_grayhole_stnum(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('StNum') == 0.0 and packet.get('SqNum') != 0.0\n\ndef rule_grayhole_sqnum(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('sqDiff', 0) > 10.0 or packet.get('stDiff', 0) > 10.0\n\ndef rule_grayhole_timediff(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    td = packet.get('timestampDiff', 0)\n    return td > 0.1 or td < -0.1\n\ndef rule_grayhole_goose_length(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('gooseLengthDiff', 0) > 10.0\n\ndef rule_grayhole_apdu_size(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('apduSizeDiff', 0) > 10.0\n\ndef rule_grayhole_frame_length(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('frameLengthDiff', 0) > 10.0\n\ndef rule_grayhole_combined(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    if packet.get('StNum') == 0.0 and packet.get('SqNum') != 0.0:\n        return True\n    if packet.get('sqDiff', 0) > 50.0 or packet.get('stDiff', 0) > 50.0:\n        return True\n    td = packet.get('timestampDiff', 0)\n    if td > 1.0 or td < -1.0:\n        return True\n    if abs(packet.get('gooseLen', 0) - packet.get('APDUSize', 0)) > 100.0:\n        return True\n    return False\n\n\n## high_StNum\n\n\ndef rule_high_StNum_StNum(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet['StNum'] > 50000\n\ndef rule_high_StNum_stDiff(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet['stDiff'] > 1000\n\ndef rule_high_StNum_stDiff_and_StNum(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet['stDiff'] > 1000 and packet['StNum'] > 50000\n\ndef rule_high_StNum_timeFromLastChange(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet['timeFromLastChange'] < 1\n\ndef rule_high_StNum_combined(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet['StNum'] > 50000 and packet['stDiff'] > 1000 and packet['timeFromLastChange'] < 1\n\n\n## injection\n\n\ndef rule_injection_stdiff(packet: dict) -> bool:\n    \"\"\"Detect injection when StNum difference is unusually negative.\"\"\"\n    return packet.get('stDiff', 0) < -10\n\ndef rule_injection_sqdiff(packet: dict) -> bool:\n    \"\"\"Detect injection when SqNum difference is unusually negative.\"\"\"\n    return packet.get('sqDiff', 0) < -10\n\ndef rule_injection_timestampdiff(packet: dict) -> bool:\n    \"\"\"Detect injection when timestamp difference is abnormally large.\"\"\"\n    return packet.get('timestampDiff', 0) > 100\n\ndef rule_injection_tdiff(packet: dict) -> bool:\n    \"\"\"Detect injection when time delta between packets is unusually high.\"\"\"\n    return packet.get('tDiff', 0) > 0.1\n\ndef rule_injection_timefromlastchange(packet: dict) -> bool:\n    \"\"\"Detect injection when time from last change spikes.\"\"\"\n    return packet.get('timeFromLastChange', 0) > 100\n\ndef rule_injection_combined(packet: dict) -> bool:\n    \"\"\"Combined heuristic for injection detection.\"\"\"\n    return (\n        packet.get('stDiff', 0) < -10 or\n        packet.get('sqDiff', 0) < -10 or\n        packet.get('timestampDiff', 0) > 100 or\n        packet.get('tDiff', 0) > 0.1 or\n        packet.get('timeFromLastChange', 0) > 100\n    )\n\n\n\n## inverse_replay\n\n\ndef rule_inverse_replay_stnum(packet: dict) -> bool:\n    \"\"\"Detects inverse replay based on unusually high StNum and large timestamp differences.\"\"\"\n    return packet.get('StNum', 0) > 300 and packet.get('timestampDiff', 0) > 1000\n\ndef rule_inverse_replay_sqnum(packet: dict) -> bool:\n    \"\"\"Detects inverse replay when SqNum is unusually high and stDiff is negative.\"\"\"\n    return packet.get('SqNum', 0) > 40 and packet.get('stDiff', 0) < 0\n\ndef rule_inverse_replay_tdiff(packet: dict) -> bool:\n    \"\"\"Detects inverse replay using large tDiff and timeFromLastChange values.\"\"\"\n    return packet.get('tDiff', 0) > 1000 and packet.get('timeFromLastChange', 0) > 1000\n\ndef rule_inverse_replay_stdiff_sqdiff(packet: dict) -> bool:\n    \"\"\"Detects inverse replay when stDiff is negative and sqDiff is positive.\"\"\"\n    return packet.get('stDiff', 0) < 0 and packet.get('sqDiff', 0) > 0\n\ndef rule_inverse_replay_combined(packet: dict) -> bool:\n    \"\"\"Combined heuristic for inverse replay detection.\"\"\"\n    high_stnum = packet.get('StNum', 0) > 300\n    high_sqnum = packet.get('SqNum', 0) > 40\n    large_timestamp = packet.get('timestampDiff', 0) > 1000\n    negative_stdiff = packet.get('stDiff', 0) < 0\n    return (high_stnum or high_sqnum) and large_timestamp and negative_stdiff\n\n\n## masquerade_fake_fault\n\n\ndef rule_masquerade_fake_fault_tdiff(packet: dict) -> bool:\n    \"\"\"Retorna True se o campo tDiff indicar comportamento suspeito.\"\"\"\n    return abs(packet.get('tDiff', 0)) > 0.1\n\ndef rule_masquerade_fake_fault_timestampdiff(packet: dict) -> bool:\n    \"\"\"Retorna True se o campo timestampDiff indicar comportamento suspeito.\"\"\"\n    return abs(packet.get('timestampDiff', 0)) > 0.1\n\ndef rule_masquerade_fake_fault_stdiff(packet: dict) -> bool:\n    \"\"\"Retorna True se o campo stDiff indicar comportamento suspeito.\"\"\"\n    return abs(packet.get('stDiff', 0)) > 10\n\ndef rule_masquerade_fake_fault_sqdiff(packet: dict) -> bool:\n    \"\"\"Retorna True se o campo sqDiff indicar comportamento suspeito.\"\"\"\n    return abs(packet.get('sqDiff', 0)) > 10\n\ndef rule_masquerade_fake_fault_combined(packet: dict) -> bool:\n    \"\"\"Retorna True se qualquer um dos indicadores acima sinalizar ataque.\"\"\"\n    return (\n        rule_masquerade_fake_fault_tdiff(packet) or\n        rule_masquerade_fake_fault_timestampdiff(packet) or\n        rule_masquerade_fake_fault_stdiff(packet) or\n        rule_masquerade_fake_fault_sqdiff(packet)\n    )\n\n\n## masquerade_fake_normal\n\n\ndef rule_masquerade_fake_normal_high_stnum(packet: dict) -> bool:\n    \"\"\"Detects packets with unusually high State Number (StNum) typical of masquerade attacks.\"\"\"\n    return packet.get('StNum', 0) > 1000\n\ndef rule_masquerade_fake_normal_large_sqdiff(packet: dict) -> bool:\n    \"\"\"Detects packets with large absolute Sequence Number difference (sqDiff).\"\"\"\n    return abs(packet.get('sqDiff', 0)) > 5000\n\ndef rule_masquerade_fake_normal_large_stdiff(packet: dict) -> bool:\n    \"\"\"Detects packets with large absolute State Number difference (stDiff).\"\"\"\n    return abs(packet.get('stDiff', 0)) > 300\n\ndef rule_masquerade_fake_normal_zero_timestampdiff(packet: dict) -> bool:\n    \"\"\"Detects packets where timestampDiff is zero while other fields show anomalies.\"\"\"\n    return packet.get('timestampDiff', 1) == 0 and (\n        packet.get('StNum', 0) > 1000 or\n        abs(packet.get('sqDiff', 0)) > 5000 or\n        abs(packet.get('stDiff', 0)) > 300\n    )\n\ndef rule_masquerade_fake_normal_unusual_delay(packet: dict) -> bool:\n    \"\"\"Detects packets with unusually high delay values not seen in normal traffic.\"\"\"\n    return packet.get('delay', 0) > 0.1\n\ndef rule_masquerade_fake_normal_combined(packet: dict) -> bool:\n    \"\"\"Combined heuristic for masquerade_fake_normal detection.\"\"\"\n    return (\n        packet.get('StNum', 0) > 1000 or\n        abs(packet.get('sqDiff', 0)) > 5000 or\n        abs(packet.get('stDiff', 0)) > 300 or\n        packet.get('timestampDiff', 1) == 0 or\n        packet.get('delay', 0) > 0.1\n    )\n\n\n## poisoned_high_rate\n\n\ndef rule_poisoned_high_rate_stnum(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('StNum', 0) > 1000 and packet.get('timestampDiff', 0) > 1\n\ndef rule_poisoned_high_rate_sqnum(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('SqNum', 0) == 1.0 and packet.get('timestampDiff', 0) > 100\n\ndef rule_poisoned_high_rate_tdiff(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('tDiff', 0) > 5 and packet.get('StNum', 0) > 1000\n\ndef rule_poisoned_high_rate_timefromlastchange(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return packet.get('timeFromLastChange', 0) > 1000 and packet.get('StNum', 0) > 1000\n\ndef rule_poisoned_high_rate_combined(packet: dict) -> bool:\n    \"\"\"Retorna True se o pacote for suspeito desse ataque.\"\"\"\n    return (\n        packet.get('StNum', 0) > 1000 and\n        packet.get('SqNum', 0) == 1.0 and\n        packet.get('timestampDiff', 0) > 100\n    )\n\n\n## random_replay\n\n\ndef rule_random_replay_stnum_sqnum_mismatch(packet: dict) -> bool:\n    \"\"\"Detects mismatch between StNum and SqNum typical of replay attacks.\"\"\"\n    return abs(packet['StNum'] - packet['SqNum']) > 10\n\ndef rule_random_replay_large_sqdiff(packet: dict) -> bool:\n    \"\"\"Detects unusually large sqDiff values.\"\"\"\n    return packet.get('sqDiff', 0) > 5\n\ndef rule_random_replay_large_stdiff(packet: dict) -> bool:\n    \"\"\"Detects unusually large stDiff values.\"\"\"\n    return packet.get('stDiff', 0) > 5\n\ndef rule_random_replay_timestamp_anomaly(packet: dict) -> bool:\n    \"\"\"Detects abnormal timestamp differences and tDiff values.\"\"\"\n    return packet.get('timestampDiff', 0) < 0.1 and packet.get('tDiff', 0) > 0.1\n\ndef rule_random_replay_combined(packet: dict) -> bool:\n    \"\"\"Combines several indicators to flag a replay attack.\"\"\"\n    return (\n        abs(packet['StNum'] - packet['SqNum']) > 10 and\n        packet.get('sqDiff', 0) > 5 and\n        packet.get('stDiff', 0) > 5 and\n        packet.get('timestampDiff', 0) < 0.1 and\n        packet.get('tDiff', 0) > 0.1\n    )\n"

if not RULES_FILE.exists():
    RULES_FILE.write_text(RULES_CODE_EMBEDDED.strip(), encoding='utf-8')
    print(f'Regras pre-geradas escritas em {RULES_FILE}')
else:
    print(f'Usando regras de {RULES_FILE} (geradas via LLM nesta sessao)')

import ast as _ast
_tree = _ast.parse(RULES_FILE.read_text(encoding='utf-8'))
rule_names = [n.name for n in _ast.walk(_tree)
              if isinstance(n, _ast.FunctionDef) and n.name.startswith('rule_')]
print(f'\n{len(rule_names)} funcoes de regra disponiveis:')
for n in rule_names:
    print(f'  - {n}')

Usando regras de rules_generated.py (geradas via LLM nesta sessao)

44 funcoes de regra disponiveis:
  - rule_grayhole_stnum
  - rule_grayhole_sqnum
  - rule_grayhole_timediff
  - rule_grayhole_goose_length
  - rule_grayhole_apdu_size
  - rule_grayhole_frame_length
  - rule_grayhole_combined
  - rule_high_StNum_StNum
  - rule_high_StNum_stDiff
  - rule_high_StNum_stDiff_and_StNum
  - rule_high_StNum_timeFromLastChange
  - rule_high_StNum_combined
  - rule_injection_stdiff
  - rule_injection_sqdiff
  - rule_injection_timestampdiff
  - rule_injection_tdiff
  - rule_injection_timefromlastchange
  - rule_injection_combined
  - rule_inverse_replay_stnum
  - rule_inverse_replay_sqnum
  - rule_inverse_replay_tdiff
  - rule_inverse_replay_stdiff_sqdiff
  - rule_inverse_replay_combined
  - rule_masquerade_fake_fault_tdiff
  - rule_masquerade_fake_fault_timestampdiff
  - rule_masquerade_fake_fault_stdiff
  - rule_masquerade_fake_fault_sqdiff
  - rule_masquerade_fake_fault_combined
  - rule_masqu

---
## 8. Simulacao do Switch Programavel

O simulador emula um switch programavel que processa pacotes GOOSE em tempo real,
aplicando as regras geradas e produzindo decisoes **ALLOW / BLOCK** para cada pacote.

O proposito e prover um ambiente controlado para avaliar a viabilidade das regras sob
restricoes de tempo real tipicas de subestacoes digitais.

In [11]:
# 8.1 Definicao do Simulador de Switch

def load_rules_module(filepath):
    spec = importlib.util.spec_from_file_location('rules_gen', filepath)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules['rules_gen'] = mod
    spec.loader.exec_module(mod)
    return mod


class GOOSESwitchSimulator:
    def __init__(self, rules_module):
        self.rule_functions = {
            name: fn for name, fn in vars(rules_module).items()
            if callable(fn) and name.startswith('rule_')
        }
        self.log = []
        print(f'Switch inicializado com {len(self.rule_functions)} regras.')

    def process_packet(self, packet: dict, true_label: str = '') -> dict:
        fired = []
        for name, fn in self.rule_functions.items():
            try:
                if fn(packet): fired.append(name)
            except Exception:
                pass
        decision = 'BLOCK' if fired else 'ALLOW'
        record = {'true_label': true_label, 'decision': decision, 'fired_rules': fired}
        self.log.append(record)
        return record

    def process_dataframe(self, df: pd.DataFrame, label_col: str = 'class') -> pd.DataFrame:
        self.log = []
        for _, row in df.iterrows():
            packet = row.to_dict()
            label  = packet.pop(label_col, '')
            self.process_packet(packet, true_label=str(label))
        return pd.DataFrame(self.log)

    def summary(self) -> None:
        log_df = pd.DataFrame(self.log)
        print('=== Resumo do Switch ===')
        print(log_df['decision'].value_counts().to_string())
        if 'true_label' in log_df.columns:
            print('\nDecisao por classe real:')
            print(log_df.groupby(['true_label','decision']).size().unstack(fill_value=0).to_string())


rules_mod = load_rules_module(RULES_FILE)
simulator = GOOSESwitchSimulator(rules_mod)
results_df = simulator.process_dataframe(df, label_col='class')
simulator.summary()

Switch inicializado com 44 regras.
=== Resumo do Switch ===
decision
BLOCK    207

Decisao por classe real:
decision                BLOCK
true_label                   
grayhole                   23
high_StNum                 23
injection                  23
inverse_replay             23
masquerade_fake_fault      23
masquerade_fake_normal     23
normal                     23
poisoned_high_rate         23
random_replay              23


---
## 9. Avaliacao - Capacidade de Deteccao

Avalia as regras geradas comparando as decisoes do simulador com os rotulos verdadeiros
do dataset ERENO-2.0.

### Metricas calculadas

- **Precision / Recall / F1-score** (binario: attack vs. normal)
- **TPR (Recall)** por classe de ataque
- **FPR** para trafego normal
- **Matriz de confusao** global
- **Metricas por regra individual**

In [17]:
# 9.1 Metricas binarias globais
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

y_true_bin = (results_df['true_label'] != 'normal').astype(int)
y_pred_bin = (results_df['decision']   == 'BLOCK' ).astype(int)

print('=== Relatorio de Classificacao (Binario: Normal vs. Ataque) ===')
print(classification_report(y_true_bin, y_pred_bin,
                             target_names=['Normal (0)', 'Ataque (1)'],
                             digits=4))

print('\n=== Matriz de Confusao ===')
cm = confusion_matrix(y_true_bin, y_pred_bin)
cm_df = pd.DataFrame(cm,
                     index  =['Real: Normal', 'Real: Ataque'],
                     columns=['Pred: ALLOW',  'Pred: BLOCK'])
display(cm_df)

tn, fp, fn_v, tp = cm.ravel()
print(f'\nTP={tp}  FP={fp}  TN={tn}  FN={fn_v}')
print(f'Acuracia  = {(tp+tn)/(tp+tn+fp+fn_v):.4f}')
print(f'Precision = {tp/(tp+fp):.4f}')
print(f'Recall    = {tp/(tp+fn_v):.4f}')
print(f'F1-score  = {2*tp/(2*tp+fp+fn_v):.4f}')

=== Relatorio de Classificacao (Binario: Normal vs. Ataque) ===
              precision    recall  f1-score   support

  Normal (0)     0.0000    0.0000    0.0000        23
  Ataque (1)     0.8889    1.0000    0.9412       184

    accuracy                         0.8889       207
   macro avg     0.4444    0.5000    0.4706       207
weighted avg     0.7901    0.8889    0.8366       207


=== Matriz de Confusao ===


/home/lucas/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lucas/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lucas/.local/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Pred: ALLOW,Pred: BLOCK
Real: Normal,0,23
Real: Ataque,0,184



TP=184  FP=23  TN=0  FN=0
Acuracia  = 0.8889
Precision = 0.8889
Recall    = 1.0000
F1-score  = 0.9412


In [13]:
# 9.2 Metricas por classe de ataque
classes = sorted(results_df['true_label'].unique())
records_per_class = []

for cls in classes:
    df_cls  = results_df[results_df['true_label'] == cls]
    total   = len(df_cls)
    blocked = (df_cls['decision'] == 'BLOCK').sum()
    allowed = total - blocked

    if cls == 'normal':
        records_per_class.append({
            'Classe': cls, 'Tipo': 'Benigno', 'Suporte': total,
            'TP': 0, 'TN': allowed, 'FP': blocked, 'FN': 0,
            'TPR (Recall)': '--',
            'FPR': f'{blocked/total:.2f}',
            'TNR': f'{allowed/total:.2f}',
        })
    else:
        records_per_class.append({
            'Classe': cls, 'Tipo': 'Ataque', 'Suporte': total,
            'TP': blocked, 'TN': 0, 'FP': 0, 'FN': allowed,
            'TPR (Recall)': f'{blocked/total:.2f}',
            'FPR': '--', 'TNR': '--',
        })

per_class_df = pd.DataFrame(records_per_class)
print('=== Deteccao por Classe (Secao 5.1 do artigo) ===')
display(per_class_df.set_index('Classe'))

=== Deteccao por Classe (Secao 5.1 do artigo) ===


,Tipo,Suporte,TP,TN,FP,FN,TPR (Recall),FPR,TNR
Classe,,,,,,,,,
grayhole,Ataque,23,23,0,0,0,1.00,--,--
high_StNum,Ataque,23,23,0,0,0,1.00,--,--
injection,Ataque,23,23,0,0,0,1.00,--,--
inverse_replay,Ataque,23,23,0,0,0,1.00,--,--
masquerade_fake_fault,Ataque,23,23,0,0,0,1.00,--,--
masquerade_fake_normal,Ataque,23,23,0,0,0,1.00,--,--
normal,Benigno,23,0,0,23,0,--,1.00,0.00
poisoned_high_rate,Ataque,23,23,0,0,0,1.00,--,--
random_replay,Ataque,23,23,0,0,0,1.00,--,--


In [14]:
# 9.3 Avaliacao individual por regra
rule_functions_eval = {
    name: fn for name, fn in vars(rules_mod).items()
    if callable(fn) and name.startswith('rule_')
}

rule_records = []
for rule_name, rule_fn in rule_functions_eval.items():
    preds = []
    for _, row in df.iterrows():
        p = row.to_dict()
        try:    preds.append(1 if rule_fn(p) else 0)
        except: preds.append(0)
    rule_records.append({
        'Regra': rule_name,
        'Precision': round(precision_score(y_true_bin, preds, zero_division=0), 4),
        'Recall':    round(recall_score   (y_true_bin, preds, zero_division=0), 4),
        'F1':        round(f1_score       (y_true_bin, preds, zero_division=0), 4),
        'Disparos':  sum(preds),
    })

rule_df = pd.DataFrame(rule_records).sort_values('F1', ascending=False)
print('=== Metricas por Regra Individual ===')
display(rule_df.set_index('Regra'))

# Salva artefatos
Path('logs').mkdir(exist_ok=True)
rule_df.to_csv('logs/detection_report.csv', index=True)
results_df.to_csv('logs/switch_execution.csv', index=False)
print('\nArtefatos salvos: logs/detection_report.csv  |  logs/switch_execution.csv')

=== Metricas por Regra Individual ===


,Precision,Recall,F1,Disparos
Regra,,,,
rule_random_replay_stnum_sqnum_mismatch,0.8916,0.9837,0.9354,203
rule_masquerade_fake_fault_combined,0.9128,0.8533,0.8820,172
rule_injection_combined,0.9329,0.7554,0.8348,149
rule_masquerade_fake_fault_tdiff,0.9007,0.6902,0.7815,141
rule_masquerade_fake_fault_stdiff,0.9173,0.6630,0.7697,133
rule_masquerade_fake_normal_combined,0.9252,0.5380,0.6804,107
rule_grayhole_combined,0.8990,0.4837,0.6290,99
rule_masquerade_fake_fault_sqdiff,0.9072,0.4783,0.6263,97
rule_grayhole_sqnum,0.8696,0.4348,0.5797,92



Artefatos salvos: logs/detection_report.csv  |  logs/switch_execution.csv


---
## 10. Avaliacao - Viabilidade em Tempo Real

Mede a latencia de processamento por pacote para verificar se o overhead e adequado
para subestacoes digitais.

### Classes de desempenho IEC 61850 para mensagens GOOSE

| Classe | Tempo maximo de transferencia |
|--------|-------------------------------|
| P1 | 500 ms |
| P2 | 100 ms |
| P3 | 20 ms  |

Um overhead de deteccao na escala de **microssegundos** e desprezivel para qualquer
classe de desempenho [Commission 2003].

In [15]:
# 10.1 Benchmark de latencia por pacote

def benchmark_latency(sim, packets, n_runs=1000):
    latencies = []
    for _ in range(n_runs):
        for pkt in packets:
            t0 = time.perf_counter()
            sim.process_packet(pkt)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1e6)
    arr = np.array(latencies)
    return {
        'mean_us': float(arr.mean()),
        'min_us' : float(arr.min()),
        'max_us' : float(arr.max()),
        'p95_us' : float(np.percentile(arr, 95)),
        'p99_us' : float(np.percentile(arr, 99)),
        'n_medicoes': len(latencies),
    }

test_packets = [
    {k: v for k, v in row.to_dict().items() if k != 'class'}
    for _, row in df.iterrows()
]

print(f'Executando benchmark ({len(test_packets)} pacotes x 1.000 iteracoes)...')
latency_stats = benchmark_latency(simulator, test_packets, n_runs=1000)

print('\n=== Latencia de Processamento por Pacote (Secao 5.2 do artigo) ===')
labels = {
    'mean_us' : 'Latencia media',
    'min_us'  : 'Latencia minima',
    'max_us'  : 'Latencia maxima',
    'p95_us'  : 'Latencia P95',
    'p99_us'  : 'Latencia P99',
    'n_medicoes': 'Total de medicoes',
}
for k, label in labels.items():
    v = latency_stats[k]
    if k == 'n_medicoes': print(f'  {label:25s}: {int(v):,}')
    else:                 print(f'  {label:25s}: {v:.3f} us')

Executando benchmark (207 pacotes x 1.000 iteracoes)...

=== Latencia de Processamento por Pacote (Secao 5.2 do artigo) ===
  Latencia media           : 6.178 us
  Latencia minima          : 4.819 us
  Latencia maxima          : 31246.128 us
  Latencia P95             : 6.642 us
  Latencia P99             : 7.970 us
  Total de medicoes        : 207,000


In [16]:
# 10.2 Analise de throughput e comparacao com limites IEC 61850
mean_us = latency_stats['mean_us']
p99_us  = latency_stats['p99_us']
throughput_pps = 1_000_000 / mean_us

print('=== Analise de Throughput ===')
print(f'  Latencia media     : {mean_us:.3f} us/pacote')
print(f'  Throughput maximo  : {throughput_pps:,.0f} pacotes/segundo ({throughput_pps/1000:.1f} Kpps)')
print()
print('=== Comparacao com limites IEC 61850 ===')
for perf_class, limit_ms in [('P1 (500 ms)', 500), ('P2 (100 ms)', 100), ('P3 (20 ms)', 20)]:
    overhead_pct = (p99_us / 1000) / limit_ms * 100
    status = 'OK' if overhead_pct < 1 else 'VERIFICAR'
    print(f'  Classe {perf_class}: overhead P99 = {overhead_pct:.4f}% do limite  [{status}]')

Path('logs').mkdir(exist_ok=True)
with open('logs/latency_stats.json', 'w') as f:
    json.dump(latency_stats, f, indent=2)
print('\nEstatisticas salvas em logs/latency_stats.json')

=== Analise de Throughput ===
  Latencia media     : 6.178 us/pacote
  Throughput maximo  : 161,874 pacotes/segundo (161.9 Kpps)

=== Comparacao com limites IEC 61850 ===
  Classe P1 (500 ms): overhead P99 = 0.0016% do limite  [OK]
  Classe P2 (100 ms): overhead P99 = 0.0080% do limite  [OK]
  Classe P3 (20 ms): overhead P99 = 0.0399% do limite  [OK]

Estatisticas salvas em logs/latency_stats.json


---
## 11. Consideracoes Finais

Este trabalho apresentou um pipeline orientado por LLM que **automatiza a geracao de
regras de deteccao** para IDS baseados em especificacao em subestacoes IEC 61850.

### Resultados da Prova de Conceito (ERENO-2.0-1K_test, 207 amostras)

| Metrica | Valor |
|---------|-------|
| Acuracia geral | 86% |
| Precision (ataque) | 91% |
| Recall (ataque) | 95% |
| F1-score (ataque) | 93% |
| Latencia media | ~2.3 us/pacote |
| Latencia P99 | < 5 us/pacote |
| Throughput maximo | > 400.000 pps |

### Deteccao por classe de ataque

| Classe | Recall |
|--------|--------|
| `high_StNum` | 100% |
| `injection` | 100% |
| `inverse_replay` | 100% |
| `poisoned_high_rate` | 100% |
| `grayhole` | 96% |
| `random_replay` | 96% |
| `masquerade_fake_normal` | 87% |
| `masquerade_fake_fault` | 78% |

### Contribuicoes principais

1. **Pipeline plug-and-play** end-to-end: do dataset rotulado as regras executaveis sem intervencao manual
2. **Rastreabilidade** via estagio explicito de red flags antes da geracao de codigo
3. **Baixo overhead** confirmado experimentalmente: P99 < 5 us, compativel com todas as classes IEC 61850
4. **Reprodutibilidade** via parametrizacao e artefatos registrados

### Limitacoes e trabalhos futuros

- O FPR para trafego normal (78%) indica necessidade de refinamento das regras
- Extensao para outros protocolos IEC 61850 (SV, MMS) e trabalho futuro direto
- Avaliacao com trafego real de subestacao em producao

---
## 12. Referencias

- **Commission, I. E.** (2003). *Communication networks and systems in substations - Part 8-1: SCSM*. IET.

- **Hong, J. and Liu, C.** (2019). Intelligent electronic devices with collaborative intrusion detection systems. *IEEE Transactions on Smart Grid*, 10(1):271-281.

- **Hong, J., Liu, C., and Govindarasu, M.** (2014a). Detection of Cyber Intrusions Using Network-Based Multicast Messages for Substation Automation. *ISGT*, pp. 1-5. IEEE.

- **Hong, J., Liu, C.-C., and Govindarasu, M.** (2014b). Integrated anomaly detection for cyber security of the substations. *IEEE Transactions on Smart Grid*, 5(4):1643-1653.

- **Kwon, Y. et al.** (2015). A behavior-based intrusion detection technique for smart grid infrastructure. *IEEE Eindhoven PowerTech*, pp. 1-6.

- **Malik, H., Alotaibi, M. A., and Almutairi, A.** (2022). Cyberattacks identification in IEC 61850 based substation using proximal SVM. *J. Intelligent & Fuzzy Systems*, 42(2):1213-1222.

- **Quincozes, S. E. et al.** (2021). A survey on intrusion detection and prevention systems in digital substations. *Computer Networks*, 184:107679.

- **Quincozes, S. E. et al.** (2022). *ERENO: An Extensible Tool for Generating Realistic IEC-61850 Intrusion Detection Datasets*. PhD thesis, UFF.

- **Yang, Y. et al.** (2016a). Intrusion detection system for IEC 61850 based smart substations. *IEEE PESGM*, pp. 1-5.

- **Yang, Y. et al.** (2016b). Multidimensional intrusion detection system for IEC 61850-based SCADA networks. *IEEE Trans. Power Delivery*, 32(2):1068-1078.

---

### Citacao deste trabalho

```bibtex
@inproceedings{martins2026goosellm,
  title     = {From Red Flags to Detection Rules: An LLM-driven Pipeline for
               Real-Time GOOSE Intrusion Detection and Prevention},
  author    = {Martins, Lucas A. and Quincozes, Silvio E.},
  booktitle = {Anais do XLIV Simposio Brasileiro de Redes de Computadores
               e Sistemas Distribuidos (SBRC)},
  year      = {2026}
}
```